## SVM

### Setup and data imports

In [1]:
# Uncomment if needed
# !pip install pandas --upgrade --quiet
# !pip install numpy --upgrade --quiet
# !pip install scikit-learn --upgrade --quiet
# !pip install matplotlib --upgrade --quiet
# !pip install seaborn --upgrade --quiet
# !pip install joblib --upgrade --quiet

In [2]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from time import time

import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_score,
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    ConfusionMatrixDisplay,
)


from sklearn.dummy import DummyClassifier

In [3]:
RANDOM_STATE = 42

PLOT_COLORS = {
    "neutral":       "#2F80ED",
    "neutral_light": "#7DB7FF",
    "income_low":    "#2F80ED",
    "income_high":   "#FF8A00",
    "after":         "#00B894",
    "outlier":       "#E63946",
    "mild":          "#F2C94C",
    "severe":        "#E63946",
    "reference":     "#4A5568",
}
TARGET_PALETTE_NUMERIC = {0: PLOT_COLORS["income_low"], 1: PLOT_COLORS["income_high"]}

sns.set_theme(
    style="whitegrid",
    palette=[PLOT_COLORS["neutral"], PLOT_COLORS["income_high"],
             PLOT_COLORS["after"], PLOT_COLORS["outlier"]],
    rc={
        "axes.spines.right": False,
        "axes.spines.top":   False,
        "axes.edgecolor":    "#D7DEE8",
        "axes.linewidth":    0.8,
        "grid.color":        "#E8EDF3",
        "grid.linewidth":    0.7,
        "legend.frameon":    False,
    },
)
plt.rcParams["figure.dpi"]        = 120
plt.rcParams["figure.figsize"]    = (10, 5)
plt.rcParams["axes.titleweight"]  = "semibold"
plt.rcParams["axes.titlepad"]     = 10


def polish_axes(ax, grid_axis="y"):
    ax.grid(False)
    ax.grid(True, axis=grid_axis, alpha=0.55)
    ax.set_axisbelow(True)
    sns.despine(ax=ax)
    return ax

In [4]:
def evaluate_svm(name, model, X_tr, y_tr, X_te, y_te, kernel, C, extra="-"):
    t0 = time()
    model.fit(X_tr, y_tr)
    fit_time = round(time() - t0, 2)

    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1] if hasattr(model, "predict_proba") else None

    acc  = accuracy_score(y_te, y_pred)
    f1   = f1_score(y_te, y_pred)
    auc  = roc_auc_score(y_te, y_proba) if y_proba is not None else float("nan")
    aupr = average_precision_score(y_te, y_proba) if y_proba is not None else float("nan")

    results.loc[name] = [kernel, C, extra,
                         round(acc, 4), round(f1, 4),
                         round(auc, 4), round(aupr, 4), fit_time]
    print(f"[{name}]  Acc={acc:.4f}  F1={f1:.4f}  AUC={auc:.4f}  AUPR={aupr:.4f}  ({fit_time}s)")
    return model

In [5]:
DATA_DIR = Path(".")

PROCESSED_FILES = {
    "X_train":     DATA_DIR / "adult_X_train.csv",
    "X_test":      DATA_DIR / "adult_X_test.csv",
    "y_train":     DATA_DIR / "adult_y_train.csv",
    "y_test":      DATA_DIR / "adult_y_test.csv",
    "preprocessor": DATA_DIR / "preprocessor.joblib",
}

missing = [name for name, path in PROCESSED_FILES.items() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing file(s): " + ", ".join(missing)
        + ". Run the preprocessing notebook first."
    )

X_train = pd.read_csv(PROCESSED_FILES["X_train"], index_col=0)
X_test  = pd.read_csv(PROCESSED_FILES["X_test"],  index_col=0)
y_train = pd.read_csv(PROCESSED_FILES["y_train"], index_col=0).squeeze("columns")
y_test  = pd.read_csv(PROCESSED_FILES["y_test"],  index_col=0).squeeze("columns")

print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}  |  y_test: {y_test.shape}")
print(f"\nClass balance (train):\n{y_train.value_counts(normalize=True).round(3)}")

X_train: (26048, 73)  |  X_test: (6513, 73)
y_train: (26048,)  |  y_test: (6513,)

Class balance (train):
income
0    0.759
1    0.241
Name: proportion, dtype: float64


In [6]:
print(f"Total columns: {X_train.shape[1]}\n")
print(X_train.columns.tolist())

Total columns: 73

['num__age', 'num__education_num', 'num__hours_per_week', 'num__capital_gain_log', 'num__capital_loss_log', 'num__net_capital', 'num__age_x_hours', 'bin__sex', 'bin__capital_gain_is_99999', 'bin__has_capital_activity', 'bin__is_higher_education', 'bin__is_US', 'cat__workclass_Federal-gov', 'cat__workclass_Local-gov', 'cat__workclass_Never-worked', 'cat__workclass_Private', 'cat__workclass_Self-emp-inc', 'cat__workclass_Self-emp-not-inc', 'cat__workclass_State-gov', 'cat__workclass_Unknown', 'cat__workclass_Without-pay', 'cat__education_10th', 'cat__education_11th', 'cat__education_12th', 'cat__education_1st-4th', 'cat__education_5th-6th', 'cat__education_7th-8th', 'cat__education_9th', 'cat__education_Assoc-acdm', 'cat__education_Assoc-voc', 'cat__education_Bachelors', 'cat__education_Doctorate', 'cat__education_HS-grad', 'cat__education_Masters', 'cat__education_Preschool', 'cat__education_Prof-school', 'cat__education_Some-college', 'cat__marital_status_Divorced', 

In [7]:
X_train_svm = X_train.copy()
X_test_svm  = X_test.copy()

print(f"Features: {X_train_svm.shape[1]}")
print(f"Train samples: {X_train_svm.shape[0]}")
print(f"Test samples:  {X_test_svm.shape[0]}")

# Results tracker — one row per model configuration
results = pd.DataFrame(columns=[
    "Kernel", "C", "gamma/degree", "Accuracy", "F1", "ROC-AUC", "AUPR", "Fit time (s)"
])

Features: 73
Train samples: 26048
Test samples:  6513


### Baseline: Dummy Classifier

In [8]:
# Baseline: Dummy Classifier 
dummy = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
dummy.fit(X_train_svm, y_train)
y_pred_dummy = dummy.predict(X_test_svm)

acc  = accuracy_score(y_test, y_pred_dummy)
f1   = f1_score(y_test, y_pred_dummy, zero_division=0)
auc  = roc_auc_score(y_test, y_pred_dummy)
aupr = average_precision_score(y_test, y_pred_dummy)

results.loc["Dummy (most_frequent)"] = ["—", "—", "—", 
                                         round(acc, 4), round(f1, 4),
                                         round(auc, 4), round(aupr, 4), 0.0]
print(classification_report(y_test, y_pred_dummy, target_names=["<=50K", ">50K"]))
results

              precision    recall  f1-score   support

       <=50K       0.76      1.00      0.86      4945
        >50K       0.00      0.00      0.00      1568

    accuracy                           0.76      6513
   macro avg       0.38      0.50      0.43      6513
weighted avg       0.58      0.76      0.66      6513



,Kernel,C,gamma/degree,Accuracy,F1,ROC-AUC,AUPR,Fit time (s)
Dummy (most_frequent),—,—,—,0.7593,0.0,0.5,0.2407,0.0


### Linear SVM

We try a range of C values manually to understand the effect of regularization
before running a full grid search. class_weight='balanced' compensates for the
~75/25 class imbalance by upweighting the minority class (>50K) during training.

In [ ]:
# Linear SVM
C_values = [0.001, 0.01, 0.1, 1, 10]

for C in C_values:
    svm_linear = SVC(kernel="linear", C=C, class_weight="balanced",
                     probability=True, random_state=RANDOM_STATE)
    evaluate_svm(
        name=f"Linear SVM (C={C})",
        model=svm_linear,
        X_tr=X_train_svm, y_tr=y_train,
        X_te=X_test_svm,  y_te=y_test,
        kernel="linear", C=C, extra="—"
    )

results